## 04. Distributions

Distribution analyses and executable components from the original DP_Transformations section. Encoding and its plans are now in 07_encoding.ipynb. Nothing is added to the common pipeline.

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import boxcox, yeojohnson
from house_prices_mlops.data_tracker import DataTracker
from house_prices_mlops.pipeline import load_stage
from IPython.display import display

In [3]:
df_before = load_stage("common")
df_hp = df_before.copy()
tracker = DataTracker.load(df_before)
profile = tracker.profile
display(profile)


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
0,Id,int64,identifier,1168,0,0.000000,0.0,0.000000,1.0,1460.0,730.904966,732.5,425.369088,0.000920,-1.215850
1,MSSubClass,string,nominal,15,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MSZoning,object,nominal,5,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LotFrontage,float64,continuous,107,217,18.578767,0.0,0.000000,21.0,313.0,70.343849,70.0,24.897021,2.406787,19.454319
4,LotArea,int64,continuous,890,0,0.000000,0.0,0.000000,1300.0,215245.0,10689.642123,9600.0,10759.366198,11.958088,186.444082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,HasBasement,int64,None,2,0,0.000000,28.0,2.397260,0.0,1.0,0.976027,1.0,0.153029,-6.232060,36.901760
89,HasPool,int64,None,2,0,0.000000,1162.0,99.486301,0.0,1.0,0.005137,0.0,0.071519,13.862369,190.491446
90,HasFireplace,int64,None,2,0,0.000000,547.0,46.832192,0.0,1.0,0.531678,1.0,0.499209,-0.127131,-1.987244
91,IsRemodeled,int64,None,2,0,0.000000,601.0,51.455479,0.0,1.0,0.485445,0.0,0.500002,0.058319,-2.000027


DISTRIBUTION TRANSFORMATIONS
- Distribution comparisons
- Executable log and sqrt transformations
- Zero-magnitude variables (log1p)

### Distribution Transformations

Estas transformaciones permiten evaluar representaciones que comprimen los valores elevados y reducen su influencia en distribuciones con asimetría positiva. No garantizan normalidad ni una mejora predictiva; su utilidad deberá compararse durante Modeling.

- **log:** se mantiene para LotArea, 1stFlrSF, GrLivArea y, por separado, SalePrice, siguiendo las decisiones del original. Comprime los valores grandes más que los pequeños y permite investigar relaciones en una escala proporcional. Requiere valores positivos; su inversa es exp.
- **sqrt:** se mantiene para LotFrontage como una compresión más suave que log. Admite cero y requiere valores no negativos. La tabla de comparación permite revisar si reduce suficientemente la asimetría; su inversa, sobre ese dominio, es elevar al cuadrado.
- **log1p:** se utiliza para las variables seleccionadas con valores cero. Aplica log(1 + x), comprime magnitudes elevadas y conserva el cero. Su inversa es expm1.

Las fórmulas de log, sqrt y log1p no aprenden parámetros estadísticos. Box-Cox y Yeo-Johnson sí estiman un parámetro en la comparación exploratoria; sus resultados no se incorporan al componente ejecutable ni deben reutilizarse como parámetros ajustados antes de validación cruzada.

SalePrice es el objetivo: se transforma fuera de X y se verifica su inversa para poder volver a la escala original. Las elecciones se conservan del notebook original; la comparación antes/después permite revisarlas, sin afirmar que sean la mejor alternativa para todos los modelos.

In [4]:
# Distribution Transformations

distribution_analysis = profile.loc[
    profile['variable_type'].isin([
        'continuous',
        'discrete'
    ]),
    [
        'var',
        'skewness',
        'kurtosis',
        'min',
        'max',
        'zero_count',
        'zero_pct'
    ]
].copy()

display(distribution_analysis.sort_values('skewness',ascending=False))

,var,skewness,kurtosis,min,max,zero_count,zero_pct
75,MiscVal,22.053579,566.356098,0.0,15500.0,1124.0,96.232877
71,PoolArea,14.396098,211.049297,0.0,738.0,1162.0,99.486301
4,LotArea,11.958088,186.444082,1300.0,215245.0,0.0,0.000000
69,3SsnPorch,9.833911,111.790190,0.0,508.0,1148.0,98.287671
45,LowQualFinSF,9.199456,87.124476,0.0,572.0,1147.0,98.202055
52,KitchenAbvGr,4.445055,21.596156,0.0,3.0,1.0,0.085616
36,BsmtFinSF2,4.217895,18.974357,0.0,1127.0,1037.0,88.784247
70,ScreenPorch,4.090393,18.505531,0.0,480.0,1071.0,91.695205
48,BsmtHalfBath,4.005786,15.110669,0.0,2.0,1102.0,94.349315
68,EnclosedPorch,3.163946,11.041964,0.0,552.0,1008.0,86.301370


In [5]:
distribution_candidates = [
    'LotArea',
    'LotFrontage',
    'SalePrice',
    '1stFlrSF',
    'GrLivArea'
]

# Evaluar transformaciones sin modificar df_hp
evaluation = []

for var in distribution_candidates:

    x = df_hp[var].dropna()

    # Transformaciones
    x_log = np.log(x)
    x_sqrt = np.sqrt(x)

    x_boxcox, lambda_bc = boxcox(x)
    x_yj, lambda_yj = yeojohnson(x)

    evaluation.append({
        'var': var,

        'original_skew': x.skew(),

        'log_skew': pd.Series(x_log).skew(),
        'sqrt_skew': pd.Series(x_sqrt).skew(),

        'boxcox_skew': pd.Series(x_boxcox).skew(),
        'boxcox_lambda': lambda_bc,

        'yeojohnson_skew': pd.Series(x_yj).skew(),
        'yeojohnson_lambda': lambda_yj
    })

transformation_comparison = pd.DataFrame(evaluation)

display(transformation_comparison)

,var,original_skew,log_skew,sqrt_skew,boxcox_skew,boxcox_lambda,yeojohnson_skew,yeojohnson_lambda
0,LotArea,11.958088,-0.013208,4.481175,0.002531,0.002822,0.002414,0.002693
1,LotFrontage,2.406787,-0.690602,0.501006,0.110368,0.355238,0.104486,0.343387
2,SalePrice,1.743129,0.124880,0.891052,-0.008051,-0.083647,-0.008051,-0.083655
3,1stFlrSF,1.422162,0.025978,0.599198,-0.000592,-0.025051,-0.000613,-0.026045
4,GrLivArea,1.425139,0.007078,0.620860,-0.000240,-0.006313,-0.000269,-0.007091


#### Executable Distribution Component
The selected log, sqrt and log1p transformations are constructed in modeling/distributions.py and combined in modeling/pipelines.py. This exploratory fit uses training only; model experiments must fit their complete preparation pipeline within each training fold. Missing values remain missing. SalePrice is processed separately from X. Box-Cox and Yeo-Johnson above are comparisons, not selected components.
Domains: log requires positive values, sqrt requires nonnegative values and log1p requires values greater than -1. No imputation or automatic clipping is introduced.


In [ ]:
from house_prices_mlops.dataset import get_predictors_and_target
from house_prices_mlops.modeling.pipelines import create_distribution_preprocessor
from house_prices_mlops.modeling.distributions import (
    DISTRIBUTION_TRANSFORMATIONS,
    TARGET_TRANSFORMATION,
    create_distribution_transformer,
)


X, y = get_predictors_and_target(df_before)

# 2. Ejecutar transformaciones de distribución
preprocessor = create_distribution_preprocessor()

X_transformed = preprocessor.fit_transform(X)

df_transformed = pd.DataFrame(
    X_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X.index,
)

# 3. Transformar SalePrice por separado
target_transformer = create_distribution_transformer(
    TARGET_TRANSFORMATION
)

y_transformed = target_transformer.fit_transform(
    y.to_numpy().reshape(-1, 1)
).ravel()

# 4. Verificar transformación inversa
np.testing.assert_allclose(
    target_transformer.inverse_transform(
        y_transformed.reshape(-1, 1)
    ).ravel(),
    y,
)

# 5. Reconstruir DataFrame para revisión
df_hp = df_before.copy()

for var, method in DISTRIBUTION_TRANSFORMATIONS.items():
    df_hp[var] = df_transformed[f"{method}__{var}"]

df_hp["SalePrice"] = y_transformed

# 6. Verificar integridad
assert len(df_hp) == len(df_before)
assert df_hp.index.equals(df_before.index)

display(df_hp.head())


In [8]:
transformations = {
    **DISTRIBUTION_TRANSFORMATIONS,
    "SalePrice": TARGET_TRANSFORMATION,
}

validation = []

for var, method in transformations.items():
    validation.append({
        "var": var,
        "method": method,
        "skew_before": df_before[var].skew(),
        "skew_after": df_hp[var].skew(),
        "kurt_before": df_before[var].kurtosis(),
        "kurt_after": df_hp[var].kurtosis(),
        "missing_before": df_before[var].isna().sum(),
        "missing_after": df_hp[var].isna().sum(),
    })

display(pd.DataFrame(validation))

,var,method,skew_before,skew_after,kurt_before,kurt_after,missing_before,missing_after
0,LotArea,log,11.958088,-0.013208,186.444082,5.197118,0,0
1,LotFrontage,sqrt,2.406787,0.501006,19.454319,4.641114,217,217
2,1stFlrSF,log,1.422162,0.025978,6.700061,0.234618,0,0
3,GrLivArea,log,1.425139,0.007078,5.451572,0.354009,0,0
4,MasVnrArea,log1p,2.286423,0.443930,6.675990,-1.677756,6,6
5,BsmtFinSF1,log1p,1.862132,-0.622601,13.475383,-1.473899,0,0
6,WoodDeckSF,log1p,1.587330,0.181798,3.116599,-1.891091,0,0
7,OpenPorchSF,log1p,2.331890,-0.064149,8.153588,-1.769901,0,0
8,SalePrice,log,1.743129,0.124880,5.475315,0.700245,0,0


In [9]:
for var in transformations:
    assert not np.isinf(df_hp[var].to_numpy()).any(), (
        f"{var} contiene valores infinitos"
    )

In [10]:
for var, method in transformations.items():

    tracker.log(
        var=var,
        stage="DP - Transformations - Distribution",
        action="transform_distribution",
        method=method,
        before={
            "skew": float(df_before[var].skew()),
            "kurtosis": float(df_before[var].kurtosis()),
            "missing": int(df_before[var].isna().sum()),
        },
        after={
            "skew": float(df_hp[var].skew()),
            "kurtosis": float(df_hp[var].kurtosis()),
            "missing": int(df_hp[var].isna().sum()),
        },
    )

### Zero-Magnitude Variables (log1p)

En estas variables el cero representa una magnitud registrada igual a cero y debe conservarse:

- **log(0)** produce −∞, por lo que log no puede aplicarse directamente cuando existen ceros.
- **log1p(0) = log(1 + 0) = 0**: conserva el cero y comprime los valores positivos.
- Un valor faltante sigue siendo faltante: log1p no realiza una imputación ni convierte información desconocida en ausencia.

Se ejecuta log1p sobre **MasVnrArea, BsmtFinSF1, WoodDeckSF y OpenPorchSF**, según las decisiones del original. La tabla siguiente compara sus distribuciones antes y después.

**TotalBsmtSF** aparece únicamente como candidata evaluada. El original no la incluyó entre las variables seleccionadas para log1p; se conserva esa distinción sin inventar una justificación o agregar una transformación.

Las variables seleccionadas ya fueron transformadas una sola vez por el componente de distribución y registradas en el bloque anterior. Esta sección compara los resultados sin volver a transformar ni registrar esas mismas operaciones.

In [11]:

zero_magnitude_candidates = [
    'MasVnrArea',
    'BsmtFinSF1',
    'TotalBsmtSF',
    'WoodDeckSF',
    'OpenPorchSF'
]
zero_evaluation = []

for var in zero_magnitude_candidates:

    x = df_before[var]
    transformed = np.log1p(x)

    zero_evaluation.append({
        'var': var,
        'zero_pct': (x == 0).mean() * 100,
        'skew_before': x.skew(),
        'skew_log1p': transformed.skew(),
        'kurt_before': x.kurtosis(),
        'kurt_log1p': transformed.kurtosis()
    })

zero_evaluation = pd.DataFrame(zero_evaluation)

display(zero_evaluation)

,var,zero_pct,skew_before,skew_log1p,kurt_before,kurt_log1p
0,MasVnrArea,57.705479,2.286423,0.443930,6.675990,-1.677756
1,BsmtFinSF1,31.934932,1.862132,-0.622601,13.475383,-1.473899
2,TotalBsmtSF,2.397260,1.723881,-5.274632,15.677793,29.306975
3,WoodDeckSF,52.910959,1.587330,0.181798,3.116599,-1.891091
4,OpenPorchSF,43.835616,2.331890,-0.064149,8.153588,-1.769901


In [12]:
# Already transformed once by the distribution component above.
log1p_vars = [var for var, method in DISTRIBUTION_TRANSFORMATIONS.items() if method == "log1p"]
display(df_hp[log1p_vars].skew())


MasVnrArea     0.443930
BsmtFinSF1    -0.622601
WoodDeckSF     0.181798
OpenPorchSF   -0.064149
dtype: float64

### Resolution for Review

The original resolution says SalePrice received log1p, but its transformation code applies np.log. This inconsistency is retained below as text for discussion; no finding is resolved automatically.

Original resolution (not executed):

```python
tracker.resolve_finding(
    key="sale_price_skewness",
    resolution="Resuelto en DP - Transformations - Distribution. Se aplicó una transformación log1p a SalePrice para reducir su asimetría positiva.",
)
```

### Save Decisions (Pending Review)

In [ ]:
tracker.save()